In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ML.MAchine import Xtest


In [3]:
RANDOM_STATE = 42
TEST_SIZE = 0.25

In [4]:
data = pd.read_csv('data/bank_transactions_data_2-selected-columns.csv')
print(data.shape)
data.head()

(2512, 10)


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online


In [5]:
data.drop('TransactionID', axis=1, inplace=True)

In [6]:
for col in data.columns:
    print(col, data[col].isna().sum())

AccountID 0
TransactionAmount 0
TransactionDate 0
TransactionType 0
Location 0
DeviceID 0
IP Address 0
MerchantID 0
Channel 0


In [7]:
data['AccountID'] = data['AccountID'].str[2:].astype(int)

In [8]:
data['DeviceID'] = data['DeviceID'].str[1:].astype(int)

In [9]:
data['MerchantID'] = data['MerchantID'].str[1:].astype(int)

TransactionType and CHannel OHE

In [10]:
data = pd.concat([data, pd.get_dummies(data['Channel'], prefix='Channel', drop_first=True)], axis=1)
data.head()

,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,Channel_Branch,Channel_Online
0,128,14.09,2023-04-11 16:29:14,Debit,San Diego,380,162.198.218.92,15,ATM,False,False
1,455,376.24,2023-06-27 16:44:19,Debit,Houston,51,13.149.61.4,52,ATM,False,False
2,19,126.29,2023-07-10 18:16:08,Debit,Mesa,235,215.97.143.157,9,Online,False,True
3,70,184.50,2023-05-05 16:32:11,Debit,Raleigh,187,200.13.225.150,2,Online,False,True
4,411,13.45,2023-10-16 17:51:24,Credit,Atlanta,308,65.164.3.100,91,Online,False,True


In [11]:
data.drop('Channel', axis=1, inplace=True)

In [12]:
data = pd.concat([data, pd.get_dummies(data['TransactionType'], prefix='TransactionType', drop_first=True)], axis=1)
data.drop('TransactionType', axis=1, inplace=True)
data.head()

,AccountID,TransactionAmount,TransactionDate,Location,DeviceID,IP Address,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit
0,128,14.09,2023-04-11 16:29:14,San Diego,380,162.198.218.92,15,False,False,True
1,455,376.24,2023-06-27 16:44:19,Houston,51,13.149.61.4,52,False,False,True
2,19,126.29,2023-07-10 18:16:08,Mesa,235,215.97.143.157,9,False,True,True
3,70,184.50,2023-05-05 16:32:11,Raleigh,187,200.13.225.150,2,False,True,True
4,411,13.45,2023-10-16 17:51:24,Atlanta,308,65.164.3.100,91,False,True,False


In [13]:
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)
data.head()

,AccountID,TransactionAmount,TransactionDate,Location,DeviceID,IP Address,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit
0,128,14.09,2023-04-11 16:29:14,San Diego,380,162.198.218.92,15,0,0,1
1,455,376.24,2023-06-27 16:44:19,Houston,51,13.149.61.4,52,0,0,1
2,19,126.29,2023-07-10 18:16:08,Mesa,235,215.97.143.157,9,0,1,1
3,70,184.50,2023-05-05 16:32:11,Raleigh,187,200.13.225.150,2,0,1,1
4,411,13.45,2023-10-16 17:51:24,Atlanta,308,65.164.3.100,91,0,1,0


In [14]:
data['year'] = data['TransactionDate'].apply(lambda x: x.split()[0].split('-')[0])
data['month'] = data['TransactionDate'].apply(lambda x: x.split()[0].split('-')[1])
data['day'] = data['TransactionDate'].apply(lambda x: x.split()[0].split('-')[2])
data.head()

,AccountID,TransactionAmount,TransactionDate,Location,DeviceID,IP Address,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day
0,128,14.09,2023-04-11 16:29:14,San Diego,380,162.198.218.92,15,0,0,1,2023,04,11
1,455,376.24,2023-06-27 16:44:19,Houston,51,13.149.61.4,52,0,0,1,2023,06,27
2,19,126.29,2023-07-10 18:16:08,Mesa,235,215.97.143.157,9,0,1,1,2023,07,10
3,70,184.50,2023-05-05 16:32:11,Raleigh,187,200.13.225.150,2,0,1,1,2023,05,05
4,411,13.45,2023-10-16 17:51:24,Atlanta,308,65.164.3.100,91,0,1,0,2023,10,16


In [15]:
data[['year', 'month', 'day']] = data[['year', 'month', 'day']].astype(int)

In [16]:
data['date'] = pd.to_datetime(data['TransactionDate'].apply(lambda x: x.split()[0])).dt.date

data['ip_transactions_per_day'] = data.groupby(['IP Address', 'date'])['IP Address'].transform('count')
data.drop('date', axis=1, inplace=True)
data.head()

,AccountID,TransactionAmount,TransactionDate,Location,DeviceID,IP Address,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day,ip_transactions_per_day
0,128,14.09,2023-04-11 16:29:14,San Diego,380,162.198.218.92,15,0,0,1,2023,4,11,1
1,455,376.24,2023-06-27 16:44:19,Houston,51,13.149.61.4,52,0,0,1,2023,6,27,1
2,19,126.29,2023-07-10 18:16:08,Mesa,235,215.97.143.157,9,0,1,1,2023,7,10,1
3,70,184.50,2023-05-05 16:32:11,Raleigh,187,200.13.225.150,2,0,1,1,2023,5,5,1
4,411,13.45,2023-10-16 17:51:24,Atlanta,308,65.164.3.100,91,0,1,0,2023,10,16,1


In [17]:
data.drop('TransactionDate', axis=1, inplace=True)

In [18]:
data['Location'].value_counts()

Location
Fort Worth          70
Los Angeles         69
Oklahoma City       68
Charlotte           68
Philadelphia        67
Tucson              67
Omaha               65
Miami               64
Houston             63
Detroit             63
Memphis             63
Denver              62
Mesa                61
Atlanta             61
Seattle             61
Kansas City         61
Boston              61
Chicago             60
Jacksonville        60
Colorado Springs    60
Fresno              60
San Diego           59
Raleigh             59
Austin              59
San Jose            59
San Antonio         59
Indianapolis        58
New York            58
San Francisco       57
Nashville           55
Las Vegas           55
Milwaukee           55
Virginia Beach      55
Phoenix             55
Columbus            54
Sacramento          53
Louisville          51
Baltimore           51
Dallas              49
Washington          48
El Paso             46
Portland            42
Albuquerque         41
Na

In [19]:
data['ips_per_acc'] = data.groupby('AccountID')['IP Address'].transform('count')

In [20]:
import requests
import time

In [21]:
def check_proxy(ip: str) -> int:
     url = f"http://ip-api.com/json/{ip}?fields=status,proxy,hosting,country,isp"
     response = requests.get(url, timeout=3)
     data = response.json()

     if data.get("status") != "success":
         return 0


     return int(data.get("proxy", False))

In [22]:
check_proxy("162.198.218.92")

0

In [23]:
def check_proxy_with_delay(ip: str) -> int:
    result = check_proxy(ip)
    time.sleep(1.5)
    return result

In [36]:
data['is_proxy'] = data['IP Address'].apply(check_proxy_with_delay)
data.head()

,AccountID,TransactionAmount,Location,DeviceID,IP Address,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day,ip_transactions_per_day,ips_per_acc,is_proxy
0,128,14.09,San Diego,380,162.198.218.92,15,0,0,1,2023,04,11,1,NaN,0
1,455,376.24,Houston,51,13.149.61.4,52,0,0,1,2023,06,27,1,2.0,0
2,19,126.29,Mesa,235,215.97.143.157,9,0,1,1,2023,07,10,1,7.0,0
3,70,184.50,Raleigh,187,200.13.225.150,2,0,1,1,2023,05,05,1,5.0,0
4,411,13.45,Atlanta,308,65.164.3.100,91,0,1,0,2023,10,16,1,9.0,0


In [27]:
data['is_proxy'].value_counts()

is_proxy
0    2504
1       8
Name: count, dtype: int64

In [24]:
data.drop('IP Address', axis=1, inplace=True)

In [66]:
data.head()

,AccountID,TransactionAmount,Location,DeviceID,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day,ip_transactions_per_day,ips_per_acc
0,128,14.09,San Diego,380,15,0,0,1,2023,04,11,1,7
1,455,376.24,Houston,51,52,0,0,1,2023,06,27,1,7
2,19,126.29,Mesa,235,9,0,1,1,2023,07,10,1,4
3,70,184.50,Raleigh,187,2,0,1,1,2023,05,05,1,8
4,411,13.45,Atlanta,308,91,0,1,0,2023,10,16,1,6


In [25]:
data['is_proxy'].to_csv('/home/dmr4ik/PycharmProjects/pythonProject/projects/bank_account_fraud_detect/ml/data/is_proxy.csv')

KeyError: 'is_proxy'

In [45]:
pwd

'/home/dmr4ik/PycharmProjects/pythonProject/projects/bank_account_fraud_detect/ml'

In [26]:
data = pd.concat([data, pd.read_csv('data/is_proxy.csv')], axis=1)
data.head()

,AccountID,TransactionAmount,Location,DeviceID,MerchantID,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day,ip_transactions_per_day,ips_per_acc,Unnamed: 0,is_proxy
0,128,14.09,San Diego,380,15,0,0,1,2023,4,11,1,7,0,0
1,455,376.24,Houston,51,52,0,0,1,2023,6,27,1,7,1,0
2,19,126.29,Mesa,235,9,0,1,1,2023,7,10,1,4,2,0
3,70,184.50,Raleigh,187,2,0,1,1,2023,5,5,1,8,3,0
4,411,13.45,Atlanta,308,91,0,1,0,2023,10,16,1,6,4,0


In [28]:
data.drop(['Unnamed: 0', 'AccountID', 'DeviceID', 'MerchantID'], axis=1, inplace=True)

In [29]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionAmount        2512 non-null   float64
 1   Location                 2512 non-null   str    
 2   Channel_Branch           2512 non-null   int64  
 3   Channel_Online           2512 non-null   int64  
 4   TransactionType_Debit    2512 non-null   int64  
 5   year                     2512 non-null   int64  
 6   month                    2512 non-null   int64  
 7   day                      2512 non-null   int64  
 8   ip_transactions_per_day  2512 non-null   int64  
 9   ips_per_acc              2512 non-null   int64  
 10  is_proxy                 2512 non-null   int64  
dtypes: float64(1), int64(9), str(1)
memory usage: 216.0 KB


In [30]:
location_freq = data['Location'].value_counts()
data['location_freq'] = data['Location'].map(location_freq)
data = data.drop(columns=['Location'])
data.head()

,TransactionAmount,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day,ip_transactions_per_day,ips_per_acc,is_proxy,location_freq
0,14.09,0,0,1,2023,4,11,1,7,0,59
1,376.24,0,0,1,2023,6,27,1,7,0,63
2,126.29,0,1,1,2023,7,10,1,4,0,61
3,184.50,0,1,1,2023,5,5,1,8,0,59
4,13.45,0,1,0,2023,10,16,1,6,0,61


In [31]:
data = data.sort_values(
    by=['year', 'month', 'day']
).reset_index(drop=True)
data.head()

,TransactionAmount,Channel_Branch,Channel_Online,TransactionType_Debit,year,month,day,ip_transactions_per_day,ips_per_acc,is_proxy,location_freq
0,129.94,0,0,1,2023,1,2,1,7,0,55
1,73.88,0,1,1,2023,1,2,1,11,0,68
2,655.15,0,0,1,2023,1,2,1,5,0,61
3,156.26,1,0,0,2023,1,2,2,7,0,55
4,245.67,1,0,0,2023,1,2,1,7,0,42


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

num_cols = ['TransactionAmount', 'ip_transactions_per_day', 'ips_per_acc', 'location_freq']
bin_cols = ['Channel_Branch', 'Channel_Online', 'TransactionType_Debit', 'is_proxy']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('bin', 'passthrough', bin_cols)
    ]
)

X = pd.DataFrame(preprocessor.fit_transform(data), columns=num_cols+bin_cols)
X.head()

,TransactionAmount,ip_transactions_per_day,ips_per_acc,location_freq,Channel_Branch,Channel_Online,TransactionType_Debit,is_proxy
0,-0.574377,-0.164265,0.394950,-0.655265,0.0,0.0,1.0,0.0
1,-0.766437,-0.164265,2.169752,1.381896,0.0,1.0,1.0,0.0
2,1.224977,-0.164265,-0.492451,0.284963,0.0,0.0,1.0,0.0
3,-0.484205,6.087742,0.394950,-0.655265,1.0,0.0,0.0,0.0
4,-0.177889,-0.164265,0.394950,-2.692427,1.0,0.0,0.0,0.0


In [33]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor as LOF

In [34]:
iso_for = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
lof = LOF(n_neighbors=20, contamination=0.02)

iso_for_pred = iso_for.fit_predict(X)
lof_pred = lof.fit_predict(X)

(iso_for_pred == lof_pred).mean()

np.float64(0.964968152866242)

In [36]:
pred = pd.DataFrame({"iso_for": iso_for_pred, "lof": lof_pred})
pred.head()

,iso_for,lof
0,1,1
1,1,1
2,1,1
3,-1,1
4,1,1


In [37]:
pred['iso_for'].value_counts()

iso_for
 1    2461
-1      51
Name: count, dtype: int64

In [38]:

pred['lof'].value_counts()

lof
 1    2461
-1      51
Name: count, dtype: int64

In [45]:
data.iloc[iso_for_pred != lof_pred]

is_proxy
0    81
1     7
Name: count, dtype: int64

In [91]:
X.to_csv('data/train_dataset.csv')

In [92]:
import joblib

In [93]:
joblib.dump(iso_for, 'isolation_forest.pkl')

['isolation_forest.pkl']

In [94]:
joblib.dump(preprocessor, 'preprocessor.pkl')

['preprocessor.pkl']

In [95]:
joblib.dump(location_freq.to_dict(), 'location_freq_map.pkl')

['location_freq_map.pkl']

In [97]:
joblib.dump(X.columns.tolist(), "feature_names.pkl")

['feature_names.pkl']

In [46]:
split_idx = int(len(X)*0.75)

Xtrain = X[:split_idx]
Xtest = X[split_idx:]

In [47]:
iso_for_tr = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
lof_tr = LOF(n_neighbors=20, contamination=0.02, novelty=True)

iso_for_tr.fit(Xtrain)
lof_tr.fit(Xtrain)

iso_for_test_pred = iso_for_tr.predict(Xtest)
lof_test_pred = lof_tr.predict(Xtest)



In [48]:
(iso_for_test_pred == lof_test_pred).mean()

np.float64(0.9617834394904459)

In [49]:
pred_test = pd.DataFrame({"iso_for": iso_for_test_pred, "lof": lof_test_pred})
pred_test.head()

,iso_for,lof
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1


In [50]:
pred_test['iso_for'].value_counts()


iso_for
 1    616
-1     12
Name: count, dtype: int64

In [52]:
pred_test['lof'].value_counts()


lof
 1    612
-1     16
Name: count, dtype: int64